# Generics that work across cells, not just within one

This notebook declares a generic type in one cell, adds generic methods to it in a later,
independently compiled cell, and instantiates/uses it in a third -- the same cross-cell
composition every other example in this repo relies on, just with type parameters instead of
concrete types.

Nothing special had to be done to support this: `AnalyzeCell` resolves identifiers via
`go/types`, not name matching, so a type parameter can never be confused with an existing
Registry symbol (Go itself forbids a type and a variable sharing a name in the same scope);
and a generic method declaration is re-injected into later cells as opaque source text, the
same way any other function or type is, so it never has to be specially understood by the
pointer-rewrite machinery at all.

In [ ]:
type Stack[T any] struct {
	items []T
}

## Add methods in a later, separate cell

`Stack[T]` was only a type declaration a moment ago -- this cell adds behavior to it, and the
next cell after that is the first one to actually use it.

In [ ]:
func (s *Stack[T]) Push(v T) {
	s.items = append(s.items, v)
}

func (s *Stack[T]) Len() int {
	return len(s.items)
}

## Instantiate and use it

`Stack[int]` here, but the same declarations above would work for `Stack[string]`,
`Stack[float64]`, or any other type argument -- nothing about the cells above was written for
`int` specifically.

In [ ]:
s := &Stack[int]{}
s.Push(1)
s.Push(2)
s.Push(3)
n := s.Len()
fmt.Println("Stack length:", n)

## Auto-display works on a generic-derived value too

A bare last expression is still captured as the cell's result (Jupyter's `Out[n]`) even when
its value came from a generic type's method -- as long as it's not itself a bare function/
method call, which is already a valid standalone Go statement and compiles fine without any
display rewriting.

In [ ]:
n